# מערכת סוכנים בסיסית — תרגיל בית מס׳ 4
**קורס: מערכות בבינה מלאכותית**

מחברת זו מממשת מערכת סוכנים מלאה הכוללת:
- סוכן ניתוב (Router Agent) לסיווג כוונות
- סוכן שפה מקומי (Local LLM Agent) עם fallback
- סוכן סנטימנט (Sentiment Agent) עם fallback מבוסס חוקים
- מנגנון בחירת כלים (Tool Selection)
- סוכן תזמור מרכזי (Orchestrator Agent)
- זיכרון קצר טווח (Short-Term Memory)
- סוכן ביקורת (Critic Agent)
- בדיקות אינטגרציה מסכמות

---

### חלק א׳ — הכנת סביבת העבודה במחברת
הגדרת התלויות הבסיסיות. ניתן להשתמש במודל מקומי אם מותקן, אך כלול גם מימוש חלופי פשוט כדי שהמחברת תרוץ גם ללא שירות חיצוני.

In [19]:
# Cell 1 - imports
import json
import re
from dataclasses import dataclass
from typing import Dict, Any, Optional, List

try:
    import requests
except ImportError:
    requests = None

### חלק ב׳ — הגדרת מבנה משותף לסוכנים
כל סוכן יחזיר אובייקט אחיד הכולל את שם הסוכן, התוצאה, רמת הביטחון ופרטי מטא־דאטה. מבנה אחיד מקל על סוכן התזמור לחבר בין הרכיבים.

In [20]:
# Cell 2 - shared result object
@dataclass
class AgentResult:
    agent_name: str
    output: Any
    confidence: float = 1.0
    metadata: Optional[Dict[str, Any]] = None

### חלק ג׳ — סוכן ניתוב לסיווג כוונות
סוכן הניתוב מקבל את הודעת המשתמש ומחזיר כוונה ורמת ביטחון. נשתמש תחילה במימוש דטרמיניסטי פשוט כדי לאפשר בדיקות יציבות.

In [21]:
# Cell 3 - router agent
INTENTS = ["generalChat", "analyzeReview", "summarizeText", "unknown"]

def router_agent(user_text: str) -> AgentResult:
    text = user_text.lower()
    if any(word in text for word in ["review", "sentiment"]):
        intent, conf = "analyzeReview", 0.93
    elif any(word in text for word in ["summarize", "summary"]):
        intent, conf = "summarizeText", 0.90
    elif len(text.strip()) < 4:
        intent, conf = "unknown", 0.40
    else:
        intent, conf = "generalChat", 0.82
    return AgentResult("RouterAgent", {"intent": intent, "confidence": conf}, conf)

**בדיקות לדוגמה — סוכן הניתוב:**

| קלט לדוגמה | תוצאה צפויה |
|---|---|
| Tell me a short joke | `{"intent": "generalChat", "confidence": 0.82}` |
| Analyze this review: The product is amazing | `{"intent": "analyzeReview", "confidence": 0.93}` |
| Summarize the following text | `{"intent": "summarizeText", "confidence": 0.90}` |
| ? | `{"intent": "unknown", "confidence": 0.40}` |

In [22]:
# Cell 4 - router tests
router_tests = [
    "Tell me a short joke",
    "Analyze this review: The product is amazing",
    "Summarize the following text",
    "?"
]

for text in router_tests:
    result = router_agent(text)
    print(text, "=>", result.output)

Tell me a short joke => {'intent': 'generalChat', 'confidence': 0.82}
Analyze this review: The product is amazing => {'intent': 'analyzeReview', 'confidence': 0.93}
Summarize the following text => {'intent': 'summarizeText', 'confidence': 0.9}
? => {'intent': 'unknown', 'confidence': 0.4}


### חלק ד׳ — סוכן שפה מקומי
סוכן השפה המקומי אחראי למשימות כלליות, כגון ניסוח קצר, שיחה פשוטה או סיכום בסיסי. במחברת ישנו מימוש חלופי כדי שהתרגיל יהיה ניתן להרצה בכל מחשב.

In [23]:
# Cell 5 - optional Ollama call and safe fallback
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3"

def call_ollama(prompt: str) -> Optional[str]:
    if requests is None:
        return None
    try:
        payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False}
        response = requests.post(OLLAMA_URL, json=payload, timeout=10)
        response.raise_for_status()
        return response.json().get("response", "").strip()
    except Exception:
        return None

def local_llm_agent(prompt: str) -> AgentResult:
    answer = call_ollama(prompt)
    if not answer:
        answer = "Local fallback answer: I received the request and will answer briefly."
    return AgentResult("LocalLLMAgent", answer, 0.80)

**בדיקה לדוגמה — סוכן שפה מקומי:**

| בדיקה | תוצאה צפויה |
|---|---|
| שם הסוכן | `LocalLLMAgent` |
| פלט | אם Ollama פעיל — תשובת המודל; אחרת — תשובת fallback מקומית |

In [24]:
# Cell 6 - local agent test
result = local_llm_agent("Tell me a short joke about AI agents")
print(result.agent_name)
print(result.output)

LocalLLMAgent
Local fallback answer: I received the request and will answer briefly.


### חלק ה׳ — סוכן כלי ייעודי לניתוח סנטימנט
במקום להרים שירות נפרד, נבנה את כלי הסנטימנט ישירות בתוך המחברת. ניתן להשתמש אם החבילה מותקנת, אחרת נפעיל מנגנון מבוסס חוקים.

In [25]:
# Cell 7 - sentiment agent with optional transformers
try:
    from transformers import pipeline
    sentiment_pipeline = pipeline("sentiment-analysis")
except Exception:
    sentiment_pipeline = None

def rule_based_sentiment(text: str) -> Dict[str, Any]:
    positive = ["good", "great", "amazing", "excellent", "love"]
    negative = ["bad", "terrible", "awful", "hate", "poor"]
    lower = text.lower()
    if any(w in lower for w in positive):
        return {"sentiment": "POSITIVE", "confidence": 0.85}
    if any(w in lower for w in negative):
        return {"sentiment": "NEGATIVE", "confidence": 0.85}
    return {"sentiment": "NEUTRAL", "confidence": 0.60}

def sentiment_agent(text: str) -> AgentResult:
    if sentiment_pipeline:
        result = sentiment_pipeline(text)[0]
        output = {
            "sentiment": result["label"],
            "confidence": round(float(result["score"]), 3)
        }
    else:
        output = rule_based_sentiment(text)
    return AgentResult("SentimentAnalysisAgent", output, output["confidence"])

**בדיקות לדוגמה — סוכן סנטימנט:**

| קלט | תוצאה צפויה |
|---|---|
| The product is amazing and I love it | POSITIVE עם ביטחון גבוה יחסית |
| This is terrible and very disappointing | NEGATIVE עם ביטחון גבוה יחסית |
| The product arrived yesterday | NEUTRAL או תוצאה פחות בטוחה |

In [26]:
# Cell 8 - sentiment tests
samples = [
    "The product is amazing and I love it",
    "This is terrible and very disappointing",
    "The product arrived yesterday"
]
for sample in samples:
    print(sample, "=>", sentiment_agent(sample).output)

The product is amazing and I love it => {'sentiment': 'POSITIVE', 'confidence': 0.85}
This is terrible and very disappointing => {'sentiment': 'NEGATIVE', 'confidence': 0.85}
The product arrived yesterday => {'sentiment': 'NEUTRAL', 'confidence': 0.6}


### חלק ו׳ — בחירת כלים
הסוכן אינו רק מייצר טקסט. עליו לבחור כלי מתאים לפי הכוונה ורמת הביטחון. זהו ההבדל המרכזי בין קריאה רגילה למודל לבין מערכת סוכנים.

In [27]:
# Cell 9 - tool selection
TOOLS = {
    "local_llm": "general chat and simple summarization",
    "sentiment_analyzer": "sentiment analysis for reviews",
    "cloud_fallback": "fallback when confidence is low"
}

def select_tool(intent: str, confidence: float) -> str:
    if confidence < 0.80:
        return "cloud_fallback"
    if intent == "analyzeReview":
        return "sentiment_analyzer"
    if intent in ["generalChat", "summarizeText"]:
        return "local_llm"
    return "cloud_fallback"

**בדיקות לדוגמה — בחירת כלים:**

| כוונה | ביטחון | כלי צפוי |
|---|---|---|
| generalChat | 0.82 | `local_llm` |
| analyzeReview | 0.93 | `sentiment_analyzer` |
| summarizeText | 0.90 | `local_llm` |
| unknown | 0.40 | `cloud_fallback` |

In [28]:
# Cell 10 - tool selection tests
cases = [
    ("generalChat", 0.82),
    ("analyzeReview", 0.93),
    ("summarizeText", 0.90),
    ("unknown", 0.40)
]
for intent, confidence in cases:
    print(intent, confidence, "=>", select_tool(intent, confidence))

generalChat 0.82 => local_llm
analyzeReview 0.93 => sentiment_analyzer
summarizeText 0.9 => local_llm
unknown 0.4 => cloud_fallback


### חלק ז׳ — סוכן תזמור מרכזי
סוכן התזמור מחבר את כל המערכת: הוא מקבל את הודעת המשתמש, מפעיל את סוכן הניתוב, בוחר כלי, מפעיל את הסוכן המתאים ומחזיר תשובה סופית.

In [29]:
# Cell 11 - orchestrator agent
agent_memory = {
    "last_intent": None,
    "last_tool": None,
    "last_user_message": None,
    "last_result": None
}

def cloud_fallback_agent(user_text: str) -> AgentResult:
    answer = "Fallback answer: the request is unclear or confidence is too low."
    return AgentResult("CloudFallbackAgent", answer, 0.70)

def orchestrator_agent(user_text: str) -> AgentResult:
    # Memory read: answer follow-up questions from the previous stored result
    if user_text.strip().lower().rstrip("?!.") in ["why", "למה"]:
        if agent_memory["last_intent"] == "analyzeReview" and agent_memory["last_result"] is not None:
            sentiment = str(agent_memory["last_result"]["sentiment"]).lower()
            answer = f"Explanation: {sentiment} words in the review indicate {sentiment} sentiment."
        elif agent_memory["last_result"] is not None:
            answer = f"Explanation based on previous result: {agent_memory['last_result']}"
        else:
            answer = "No relevant previous context was found."
        return AgentResult("OrchestratorAgent", answer, 0.75, {"memory_used": True})

    route = router_agent(user_text)
    intent = route.output["intent"]
    confidence = route.output["confidence"]
    tool = select_tool(intent, confidence)

    if tool == "sentiment_analyzer":
        result = sentiment_agent(user_text)
    elif tool == "local_llm":
        result = local_llm_agent(user_text)
    else:
        result = cloud_fallback_agent(user_text)

    # Critic check: if the quality score is low, fall back to the cloud agent
    # ("critic_agent" in globals() keeps Cell 12 runnable before Cell 14 defines it)
    if tool != "cloud_fallback" and "critic_agent" in globals():
        critique = critic_agent(result.output)
        if critique.output["quality_score"] < 3:
            result = cloud_fallback_agent(user_text)
            tool = "cloud_fallback"

    agent_memory.update({
        "last_intent": intent,
        "last_tool": tool,
        "last_user_message": user_text,
        "last_result": result.output
    })
    return result

**בדיקות אינטגרציה לדוגמה:**

| תרחיש | סוכן צפוי | תוצאה צפויה |
|---|---|---|
| שיחה כללית | `LocalLLMAgent` | תשובה מקומית או תשובת fallback |
| ניתוח ביקורת חיובית | `SentimentAnalysisAgent` | POSITIVE עם ביטחון גבוה |
| סיכום טקסט | `LocalLLMAgent` | תשובת סיכום או fallback |
| בקשה לא ברורה | `CloudFallbackAgent` | הודעת גיבוי |

In [30]:
# Cell 12 - integration tests
messages = [
    "Tell me a short joke",
    "Analyze this review: The product is amazing",
    "Summarize the following paragraph",
    "?"
]
for msg in messages:
    result = orchestrator_agent(msg)
    print("USER:", msg)
    print("AGENT:", result.agent_name)
    print("OUTPUT:", result.output)
    print("MEMORY:", agent_memory)
    print("---")

USER: Tell me a short joke
AGENT: LocalLLMAgent
OUTPUT: Local fallback answer: I received the request and will answer briefly.
MEMORY: {'last_intent': 'generalChat', 'last_tool': 'local_llm', 'last_user_message': 'Tell me a short joke', 'last_result': 'Local fallback answer: I received the request and will answer briefly.'}
---
USER: Analyze this review: The product is amazing
AGENT: SentimentAnalysisAgent
OUTPUT: {'sentiment': 'POSITIVE', 'confidence': 0.85}
MEMORY: {'last_intent': 'analyzeReview', 'last_tool': 'sentiment_analyzer', 'last_user_message': 'Analyze this review: The product is amazing', 'last_result': {'sentiment': 'POSITIVE', 'confidence': 0.85}}
---
USER: Summarize the following paragraph
AGENT: LocalLLMAgent
OUTPUT: Local fallback answer: I received the request and will answer briefly.
MEMORY: {'last_intent': 'summarizeText', 'last_tool': 'local_llm', 'last_user_message': 'Summarize the following paragraph', 'last_result': 'Local fallback answer: I received the request

### חלק ח׳ — זיכרון קצר טווח
הזיכרון מאפשר לסוכן להבין בקשות המשך. לדוגמה, לאחר ניתוח ביקורת, המשתמש יכול לשאול: "למה?" והמערכת תוכל להתייחס לתוצאה הקודמת.

| שלב | תוצאה צפויה |
|---|---|
| לאחר ניתוח ביקורת | בזיכרון נשמרים הכוונה, הכלי, הודעת המשתמש והתוצאה |
| בקשת המשך: למה? | המערכת משתמשת בתוצאה הקודמת כדי להסביר את הסיווג |

In [31]:
# Cell 13 - memory follow-up example
orchestrator_agent("Analyze this review: The product is amazing")
print(agent_memory)

follow_up = "Why?"
result = orchestrator_agent(follow_up)
print(result.output)

{'last_intent': 'analyzeReview', 'last_tool': 'sentiment_analyzer', 'last_user_message': 'Analyze this review: The product is amazing', 'last_result': {'sentiment': 'POSITIVE', 'confidence': 0.85}}
Explanation: positive words in the review indicate positive sentiment.


### חלק ט׳ — סוכן ביקורת
סוכן הביקורת מעריך את איכות התשובה של סוכן אחר. אם הציון נמוך מ־3, סוכן התזמור צריך להפעיל גיבוי.

| תשובה נבדקת | ציון צפוי |
|---|---|
| תשובה קצרה מאוד, למשל: `כן` | 2 — יש להפעיל גיבוי |
| תשובה מפורטת וברורה | 3 או 4 — ניתן לקבל |
| תשובת סנטימנט מובנית | 4 — בדרך כלל תקינה |

In [32]:
# Cell 14 - critic agent
def critic_agent(answer: Any) -> AgentResult:
    text = str(answer)
    lowered = text.lower()
    if len(text.strip()) < 20:
        score = 2
    elif "example" in lowered or "positive" in lowered or "negative" in lowered:
        score = 4
    else:
        score = 3
    return AgentResult("CriticAgent", {"quality_score": score}, score / 5)

# Example
answer = local_llm_agent("Say something short").output
critique = critic_agent(answer)
print(answer)
print(critique.output)

Local fallback answer: I received the request and will answer briefly.
{'quality_score': 3}


### חלק י׳ — בדיקות מסכמות במחברת
בסוף המחברת יש תא בדיקות מסכם. התא מריץ מספר תרחישים ומוודא שכל רכיבי המערכת עובדים יחד.

**תוצאה צפויה:**
```
PASS: Tell me a short joke => LocalLLMAgent
PASS: Analyze this review: This product is terrible => SentimentAnalysisAgent
PASS: ? => CloudFallbackAgent
All final tests passed
```

In [33]:
# Cell 15 - final validation
final_tests = [
    ("Tell me a short joke", "LocalLLMAgent"),
    ("Analyze this review: This product is terrible", "SentimentAnalysisAgent"),
    ("?", "CloudFallbackAgent")
]

for message, expected_agent in final_tests:
    result = orchestrator_agent(message)
    assert result.agent_name == expected_agent, (message, result.agent_name, expected_agent)
    print("PASS:", message, "=>", result.agent_name)

print("All final tests passed")

PASS: Tell me a short joke => LocalLLMAgent
PASS: Analyze this review: This product is terrible => SentimentAnalysisAgent
PASS: ? => CloudFallbackAgent
All final tests passed


---

### סיכום התרגיל

* **מה עבד היטב:** חלוקת האחריות לארכיטקטורה של סוכנים (נתב, מנתח סנטימנט, ומודל שיחה) הקלה על פיתוח המערכת וניהול הזרימה. התזמור בין הרכיבים איפשר לנתב שאילתות בדיוק למקום המתאים.

* **מה היה מוגבל:** סוכן הניתוב שלנו הסתמך על חוקיות נוקשה (if-else) שהיא מוגבלת מאוד בהשוואה להבנת שפה טבעית. כמו כן, מימוש הסנטימנט מבוסס החוקים יכול בקלות לפספס ניואנסים או משפטים מורכבים.

* **מתי כדאי להשתמש במודל מקומי לעומת שירות ענן:** כדאי להשתמש במודל מקומי כאשר יש צורך בשמירה על פרטיות הנתונים, חיסכון בעלויות API חיצוניות וכאשר המשימה פשוטה מספיק. שירות ענן עדיף כאשר נדרשים ביצועים גבוהים, זמני תגובה מהירים מאוד ללא עומס על חומרת הקצה, או כשיש צורך להשתמש במודלי שפה מתקדמים וכבדים בעלי יכולות הסקה מורכבות.